> **Course notebook 7** | [Course home](../README.md) | [Previous: Attention foundations](attention_foundations_from_rnn_to_self_attention.ipynb)

This notebook turns the previous conceptual attention lesson into small, traceable calculations.

# Simplified Self-Attention from Scratch

We will build self-attention without trainable attention parameters:

```text
query → scores → weights → weighted sum → context vector
```

The examples are deliberately small enough to calculate by hand. Query, Key, and Value projections, scaling, causal masking, multi-head attention, and Transformer blocks belong to later notebooks.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(123)
np.random.seed(123)
torch.set_printoptions(precision=4, sci_mode=False)
plt.rcParams.update({"figure.figsize": (8.5, 3.8), "font.size": 11})

COLORS = {
    "blue": "#3b82f6",
    "orange": "#f59e0b",
    "green": "#10b981",
    "red": "#ef4444",
    "purple": "#8b5cf6",
    "gray": "#64748b",
}


def print_table(headers, rows):
    '''Print a small dependency-free text table.'''
    text_rows = [[str(value) for value in row] for row in rows]
    widths = [
        max(len(str(header)), *(len(row[i]) for row in text_rows))
        for i, header in enumerate(headers)
    ]
    line = "-+-".join("-" * width for width in widths)
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print(line)
    for row in text_rows:
        print(" | ".join(value.ljust(widths[i]) for i, value in enumerate(row)))


def annotated_heatmap(values, row_labels, column_labels, title, colorbar_label,
                      cmap="Blues", value_format=".2f"):
    '''Draw a labeled heatmap with its values written inside.'''
    array = np.asarray(values)
    fig, ax = plt.subplots(figsize=(7.2, 5.5))
    image = ax.imshow(array, cmap=cmap)
    ax.set_xticks(range(len(column_labels)), column_labels, rotation=35, ha="right")
    ax.set_yticks(range(len(row_labels)), row_labels)
    ax.set_xlabel("Token being compared / combined")
    ax.set_ylabel("Query token")
    ax.set_title(title)
    threshold = (array.min() + array.max()) / 2
    for row in range(array.shape[0]):
        for col in range(array.shape[1]):
            ax.text(
                col, row, format(array[row, col], value_format),
                ha="center", va="center",
                color="white" if array[row, col] > threshold else "black",
                fontsize=9,
            )
    fig.colorbar(image, ax=ax, label=colorbar_label)
    plt.tight_layout()
    plt.show()


print("PyTorch:", torch.__version__)
print("Random seed: 123")

## 0. Where we are in the attention roadmap

```text
1. Simplified self-attention without trainable weights  ← YOU ARE HERE
2. Self-attention with trainable weights
3. Causal attention
4. Multi-head attention
```

Removing trainable parameters lets us see the core mechanics first. The mechanism will later become more powerful, but this basic pipeline will remain recognizable.

**Concept check:** Which major feature has been intentionally removed from this lesson?

## 1. Recap: why attention exists

A token embedding represents one token, but it does not explicitly answer:

> How important are the other tokens for this token in this sequence?

For **“your journey starts with one step,”** ask what *journey* should take from *your*, *starts*, *with*, *one*, and *step*.

```text
embedding vector → ATTENTION → context vector
```

- **Embedding vector:** numerical representation before contextual mixing.
- **Context vector:** enriched representation that combines information from sequence positions.

**Concept check:** Which vector exists after information from other positions has been mixed in?

## 2. From text to embeddings

```text
"your journey starts with one step"
                 ↓
["your", "journey", "starts", "with", "one", "step"]
                 ↓
               token IDs
                 ↓
            embedding lookup
                 ↓
               vectors
```

GPT tokenizers normally use subwords, so a token is not always a whole word. We treat each word as one token here only to keep the arithmetic readable.

## 3. Small 3D embeddings

The six manually chosen vectors below make the geometry visible. Real token embeddings are learned and usually have hundreds or thousands of dimensions.

> **Illustrative values for learning — not embeddings learned by a real LLM.**

In [ ]:
tokens = ["your", "journey", "starts", "with", "one", "step"]
inputs = torch.tensor([
    [0.20, 0.50, 0.30],  # your
    [0.80, 0.60, 0.40],  # journey
    [0.75, 0.55, 0.35],  # starts
    [0.10, 0.70, 0.20],  # with
    [0.00, 0.20, 0.90],  # one
    [0.65, 0.35, 0.50],  # step
], dtype=torch.float32)

rows = [
    [token, *(f"{value:.2f}" for value in vector.tolist())]
    for token, vector in zip(tokens, inputs)
]
print_table(["token", "dim_1", "dim_2", "dim_3"], rows)
print("\ninputs.shape:", tuple(inputs.shape))

**What to notice:** rows are tokens, columns are embedding dimensions, so six 3D tokens produce shape `(6, 3)`.

**Concept check:** Which axis would grow if the sentence contained eight tokens?

## 4. Visualize the embeddings in 3D

The vectors are drawn from the origin. Their directions were chosen so *journey* and *starts* align closely, *step* aligns somewhat, and *one* aligns less.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
for token, vector in zip(tokens, inputs.numpy()):
    color = COLORS["red"] if token == "journey" else COLORS["blue"]
    ax.quiver(0, 0, 0, *vector, color=color, arrow_length_ratio=0.1, lw=2)
    ax.text(*(vector + 0.025), token, color=color)
ax.set(
    xlabel="dimension 1", ylabel="dimension 2", zlabel="dimension 3",
    title="Illustrative 3D token embeddings",
    xlim=(0, 1), ylim=(0, 1), zlim=(0, 1),
)
plt.show()

**What to notice:** alignment is easier to see in 3D. Real embedding spaces are too high-dimensional to plot directly.

**Concept check:** Which vector appears most aligned with *journey*?

## 5. What is the query?

We first calculate context only for *journey*. In this simplified lesson, the **query** is the token representation whose context vector we are currently calculating.

```text
your      x₁
journey   x₂  ← QUERY
starts    x₃
with      x₄
one       x₅
step      x₆

target: calculate z₂
```

Python uses zero-based indexing, so the second token is `inputs[1]`.

In [ ]:
query_index = 1
query = inputs[query_index]

print("Query token:", tokens[query_index])
print("query = inputs[1] =", query)
print("query.shape:", tuple(query.shape))

**What to notice:** the query is one 3D row, so its shape is `(3,)`.

**Concept check:** Will *journey* remain the only query when we generalize to the full sequence?

## 6. What information do we need?

To construct `z₂`, we need a contribution strength for every pair:

```text
journey ↔ your       journey ↔ journey
journey ↔ starts     journey ↔ with
journey ↔ one        journey ↔ step
```

An **attention score** is a raw numerical compatibility measure between the query and another input representation.

**Concept check:** How many scores are needed for one query in this six-token sentence?

## 7. Dot-product intuition

The simplified score uses a dot product:

\[
a \cdot b = \lVert a \rVert \lVert b \rVert \cos(\theta)
\]

Similar directions have a smaller angle and usually a larger dot product. Approximately perpendicular directions have cosine near zero.

In [ ]:
vector_a = np.array([1.0, 0.35])
vector_b = np.array([0.85, 0.45])   # roughly aligned
vector_c = np.array([-0.35, 1.0])   # perpendicular to A

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, second, label, color in [
    (axes[0], vector_b, "B: aligned", COLORS["green"]),
    (axes[1], vector_c, "C: perpendicular", COLORS["orange"]),
]:
    ax.quiver(0, 0, *vector_a, angles="xy", scale_units="xy", scale=1,
              color=COLORS["blue"], width=0.015, label="A")
    ax.quiver(0, 0, *second, angles="xy", scale_units="xy", scale=1,
              color=color, width=0.015, label=label)
    ax.set(xlim=(-0.6, 1.3), ylim=(-0.2, 1.3), xlabel="dimension 1",
           ylabel="dimension 2", aspect="equal")
    ax.legend()
    ax.set_title(f"A · {label[0]} = {np.dot(vector_a, second):.3f}")
fig.suptitle("Vector direction affects the dot product")
plt.tight_layout()
plt.show()

**Precision note:** a dot product is not pure cosine similarity; vector magnitudes also affect it. Here it is an intuitive compatibility measure.

**Concept check:** Which pair has the larger dot product?

## 8. Attention score for one pair

Start with the compatibility between the *journey* query and the *your* vector:

\[
score(journey, your) = journey \cdot your
\]

In [ ]:
x1 = inputs[0]
products = query * x1

for dimension, (q_value, x_value, product) in enumerate(
    zip(query, x1, products), start=1
):
    print(
        f"dimension {dimension}: {q_value.item():.2f} × "
        f"{x_value.item():.2f} = {product.item():.3f}"
    )

manual_pair_score = products.sum()
torch_pair_score = torch.dot(query, x1)
print("\nmanual sum:", manual_pair_score.item())
print("torch.dot: ", torch_pair_score.item())
print("match:", torch.allclose(manual_pair_score, torch_pair_score))

**What to notice:** multiply matching dimensions, then add the products.

**Concept check:** How many multiplications are used for two 3D vectors?

## 9. Attention scores for all tokens

We now compare *journey* with every input vector. A loop keeps the six dot products visible.

In [ ]:
attention_scores_2 = torch.empty(len(tokens))

for i, x_i in enumerate(inputs):
    attention_scores_2[i] = torch.dot(query, x_i)

print_table(
    ["token", "raw attention score"],
    [[token, f"{score.item():.4f}"] for token, score in zip(tokens, attention_scores_2)],
)

colors = [COLORS["red"] if token == "journey" else COLORS["blue"] for token in tokens]
plt.bar(tokens, attention_scores_2.numpy(), color=colors)
plt.xlabel("Token compared with the journey query")
plt.ylabel("Raw dot-product score")
plt.title("Raw attention scores for journey")
plt.xticks(rotation=25)
plt.show()

**What to notice:** higher means greater raw compatibility. The self-score may be large because it is the vector's squared magnitude.

**Concept check:** Which non-query token has the highest raw score?

## 10. Connect the scores to the geometry

Sort the scores and compare the order with the 3D plot.

In [ ]:
sorted_indices = torch.argsort(attention_scores_2, descending=True)
print_table(
    ["rank", "token", "journey dot product"],
    [
        [rank, tokens[index], f"{attention_scores_2[index].item():.4f}"]
        for rank, index in enumerate(sorted_indices.tolist(), start=1)
    ],
)

**What to notice:** the artificial geometry and dot products agree by construction. This is not sufficient for real language understanding; raw similarity can miss task-specific relationships.

**Concept check:** Is this ranking learned from text?

## 11. Scores are not weights

| Attention score | Attention weight |
|---|---|
| Raw compatibility number | Normalized relative contribution |
| Need not sum to 1 | Weights for one query sum to 1 |
| May have arbitrary scale or sign | Positive after softmax |

```text
dot products → scores → NORMALIZATION → weights
```

**Concept check:** Which quantity is used directly in the weighted combination?

## 12. Simple sum normalization

Before softmax, consider positive scores `[1, 5, 2]`. Divide each value by their sum.

In [ ]:
simple_scores = torch.tensor([1.0, 5.0, 2.0])
score_sum = simple_scores.sum()
simple_weights = simple_scores / score_sum

for score, weight in zip(simple_scores, simple_weights):
    print(f"{score.item():.0f} / {score_sum.item():.0f} = {weight.item():.3f}")
print("sum:", simple_weights.sum().item())

**What to notice:** `[0.125, 0.625, 0.250]` is easy to interpret as relative contributions for these positive values.

**Concept check:** What must be true of the normalized sum?

## 13. Why direct division is limited

Directly dividing `[-2, 1, 4]` by its sum produces a negative “weight” and another weight above 1. If the scores sum to zero, division is undefined.

In [ ]:
awkward_scores = torch.tensor([-2.0, 1.0, 4.0])
awkward_result = awkward_scores / awkward_scores.sum()
print("scores:", awkward_scores)
print("direct division:", awkward_result)
print("sum:", awkward_result.sum().item())

**What to notice:** summing to 1 alone does not make values useful probability-like contributions. Standard attention uses **softmax**.

**Concept check:** Which result makes direct division awkward here?

## 14. Softmax intuition

\[
softmax(s_i) = \frac{\exp(s_i)}{\sum_j \exp(s_j)}
\]

Steps: exponentiate each score, sum the exponentials, then divide each exponential by that total.

In [ ]:
exponentials = torch.exp(simple_scores)
softmax_manual = exponentials / exponentials.sum()

print_table(
    ["raw score", "exp(score)", "softmax weight"],
    [
        [f"{score.item():.0f}", f"{exp_value.item():.4f}", f"{weight.item():.4f}"]
        for score, exp_value, weight in zip(simple_scores, exponentials, softmax_manual)
    ],
)
print("\nweight sum:", softmax_manual.sum().item())

**What to notice:** exponentials are positive, and division by their total makes the outputs sum to 1.

**Concept check:** Which raw score receives the largest softmax weight?

## 15. What softmax does to differences

Compare raw scores, direct positive-score normalization, and softmax.

In [ ]:
comparison = torch.stack([simple_scores, simple_weights, softmax_manual])
labels = ["raw scores", "simple normalization", "softmax"]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
for ax, values, label in zip(axes, comparison, labels):
    ax.bar(["1", "5", "2"], values.numpy(), color=COLORS["blue"])
    ax.set_xlabel("Original score label")
    ax.set_ylabel("Value")
    ax.set_title(label)
fig.suptitle("Softmax emphasizes relative score differences")
plt.tight_layout()
plt.show()

**What to notice:** softmax assigns approximately `[0.017, 0.936, 0.047]`. Concentration depends on score differences; softmax does not always make one value nearly 1.

**Concept check:** Which operation magnifies relative differences here?

## 16. Important softmax properties

1. Every output is positive because `exp(x) > 0`.
2. Outputs sum to 1.
3. Larger scores receive larger weights.
4. Adding one constant to every score does not change the result.

In [ ]:
original_softmax = torch.softmax(torch.tensor([1.0, 5.0, 2.0]), dim=0)
shifted_softmax = torch.softmax(torch.tensor([101.0, 105.0, 102.0]), dim=0)

print("softmax([1, 5, 2]):      ", original_softmax)
print("softmax([101, 105, 102]):", shifted_softmax)
print("approximately equal:", torch.allclose(original_softmax, shifted_softmax))
print("positive:", bool(torch.all(original_softmax > 0)))
print("sum:", original_softmax.sum().item())

**What to notice:** only relative differences matter, not a shared offset.

**Concept check:** Does adding 100 change any pairwise score difference?

## 17. Numerical stability

Directly computing very large exponentials can overflow. Subtract the maximum before exponentiation:

```text
[1000, 999, 998] - 1000 = [0, -1, -2]
```

In [ ]:
large_scores_np = np.array([1000.0, 999.0, 998.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive_exp = np.exp(large_scores_np)
    naive_weights_np = naive_exp / naive_exp.sum()

stable_scores_np = large_scores_np - large_scores_np.max()
stable_exp = np.exp(stable_scores_np)
stable_weights_np = stable_exp / stable_exp.sum()

print("naive exp:", naive_exp)
print("naive result:", naive_weights_np)
print("shifted scores:", stable_scores_np)
print("stable exp:", stable_exp)
print("stable result:", stable_weights_np)

**What to notice:** the naive calculation becomes `inf/inf`, while `[0, -1, -2]` exponentiates safely.

**Concept check:** What value becomes zero after subtracting the maximum?

## 18. Why subtracting the maximum changes nothing

For `m = max(s)`:

\[
\frac{\exp(s_i-m)}{\sum_j \exp(s_j-m)}
=
\frac{\exp(s_i)/\exp(m)}{\sum_j \exp(s_j)/\exp(m)}
\]

The common factor cancels.

> **Same mathematical result; much better numerical stability.**

**Concept check:** Is the ranking of scores changed by subtracting the same value?

## 19. Implement softmax three ways

Compare naive, stable, and PyTorch implementations on ordinary and large inputs.

In [ ]:
def naive_softmax(values):
    exponentials = torch.exp(values)
    return exponentials / exponentials.sum()


def stable_softmax(values):
    shifted = values - values.max()
    exponentials = torch.exp(shifted)
    return exponentials / exponentials.sum()


normal = torch.tensor([1.0, 5.0, 2.0])
large = torch.tensor([1000.0, 999.0, 998.0])

method_rows = []
for name, function in [
    ("naive_softmax", naive_softmax),
    ("stable_softmax", stable_softmax),
    ("torch.softmax", lambda x: torch.softmax(x, dim=0)),
]:
    normal_result = function(normal)
    large_result = function(large)
    method_rows.append([
        name,
        str(normal_result.tolist()),
        str(large_result.tolist()),
        str(bool(torch.isfinite(large_result).all())),
    ])

print_table(["method", "normal input", "large input", "stable?"], method_rows)

**What to notice:** `torch.softmax` is stable and optimized, so it should normally be used in PyTorch code.

**Concept check:** Which custom implementation applies the subtract-max trick?

## 20. Apply softmax to journey's scores

Convert the six raw scores into relative contribution weights.

In [ ]:
attention_weights_2 = torch.softmax(attention_scores_2, dim=0)

print_table(
    ["token", "raw score", "attention weight", "percentage"],
    [
        [
            token,
            f"{score.item():.4f}",
            f"{weight.item():.4f}",
            f"{100 * weight.item():.1f}%",
        ]
        for token, score, weight in zip(tokens, attention_scores_2, attention_weights_2)
    ],
)
print("\nweight sum:", attention_weights_2.sum().item())

plt.bar(tokens, attention_weights_2.numpy(), color=COLORS["orange"])
plt.xlabel("Token contributing to journey")
plt.ylabel("Softmax attention weight")
plt.title("Normalized attention weights for journey")
plt.xticks(rotation=25)
plt.show()

**Illustrative values for learning — not weights learned by a real LLM.**

**What to notice:** *journey* gives relatively more weight to *journey/starts* and less to lower-scoring tokens. These percentages are not linguistic truth.

**Concept check:** Why do the bars sum to 1?

## 21. Scores vs weights

```text
raw dot-product scores
          ↓ softmax
positive normalized weights
```

| Score | Weight |
|---|---|
| Raw compatibility | Relative contribution |
| Does not need to sum to 1 | One query's weights sum to 1 |
| Used as softmax input | Used in the weighted sum |

**Concept check:** Which quantity is produced directly by `torch.dot`?

## 22. From weights to a context vector

For *journey*:

\[
z_2 = \alpha_{21}x_1 + \alpha_{22}x_2 + \cdots + \alpha_{26}x_6
\]

```text
x1 × α21 ─┐
x2 × α22 ─┤
x3 × α23 ─┤
x4 × α24 ─┼── sum → z2
x5 × α25 ─┤
x6 × α26 ─┘
```

Each input vector is scaled by its weight, then the scaled vectors are added.

**Concept check:** What determines how strongly `x3` contributes?

## 23. Visualize vector scaling

Compare each original vector with its weighted contribution.

In [ ]:
scaled_vectors_2 = attention_weights_2[:, None] * inputs
context_from_scaled = scaled_vectors_2.sum(dim=0)

print_table(
    ["token", "original vector", "weight", "scaled vector"],
    [
        [
            token,
            str([round(v, 3) for v in original.tolist()]),
            f"{weight.item():.4f}",
            str([round(v, 3) for v in scaled.tolist()]),
        ]
        for token, original, weight, scaled in zip(
            tokens, inputs, attention_weights_2, scaled_vectors_2
        )
    ],
)

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
for token, original, scaled in zip(tokens, inputs.numpy(), scaled_vectors_2.numpy()):
    ax.quiver(0, 0, 0, *original, color=COLORS["gray"], alpha=0.25,
              arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, *scaled, color=COLORS["blue"],
              arrow_length_ratio=0.1, lw=2)
    ax.text(*scaled, token, fontsize=9)
ax.quiver(0, 0, 0, *context_from_scaled.numpy(), color=COLORS["red"],
          arrow_length_ratio=0.1, lw=4)
ax.text(*context_from_scaled.numpy(), " z₂", color=COLORS["red"], weight="bold")
ax.set(
    xlabel="dimension 1", ylabel="dimension 2", zlabel="dimension 3",
    title="Original vectors (light), weighted contributions, and z₂",
    xlim=(0, 1), ylim=(0, 1), zlim=(0, 1),
)
plt.show()

**Illustrative values for learning — not embeddings learned by a real LLM.**

**What to notice:** a larger attention weight creates a stronger contribution to the final red vector.

**Concept check:** Are the blue vectors new independent embeddings or scaled inputs?

## 24. Calculate `z₂` manually

Use a loop so every weighted contribution remains visible.

In [ ]:
context_vec_2 = torch.zeros(inputs.shape[1])

for token, weight, x_i in zip(tokens, attention_weights_2, inputs):
    contribution = weight * x_i
    context_vec_2 += contribution
    print(
        f"{token:>7}: {weight.item():.4f} × {x_i.tolist()} "
        f"= {[round(v, 4) for v in contribution.tolist()]}"
    )

print("\nz₂ =", context_vec_2)

**What to notice:** `z₂` contains weighted information from the whole sequence, not only the *journey* row.

**Concept check:** Why does `z₂` still have three dimensions?

## 25. Embedding vector vs context vector

- `x₂`: *journey* before contextual mixing.
- `z₂`: *journey* after the simplified weighted mixture.

In [ ]:
print("x₂:", query)
print("z₂:", context_vec_2)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
for vector, label, color in [
    (query.numpy(), "x₂: original journey", COLORS["blue"]),
    (context_vec_2.numpy(), "z₂: context", COLORS["red"]),
]:
    ax.quiver(0, 0, 0, *vector, color=color, arrow_length_ratio=0.1, lw=3)
    ax.text(*vector, f" {label}", color=color)
ax.set(
    xlabel="dimension 1", ylabel="dimension 2", zlabel="dimension 3",
    title="Journey before and after simplified contextual mixing",
    xlim=(0, 1), ylim=(0, 1), zlim=(0, 1),
)
plt.show()

**What to notice:** the vectors differ because `z₂` includes all weighted inputs. This toy result is not equivalent to a rich representation learned by a modern Transformer.

**Concept check:** Which vector came directly from the embedding table?

## 26. Every token needs a context vector

```text
your    → z₁       with → z₄
journey → z₂       one  → z₅
starts  → z₃       step → z₆
```

Every token takes a turn acting as the query. Six queries will produce six context vectors.

**Concept check:** How many score rows will the complete calculation need?

## 27. Scores for every query

Begin with nested loops: each query is compared with all six input vectors.

In [ ]:
attention_scores_loop = torch.empty((len(tokens), len(tokens)))

for query_i, query_vector in enumerate(inputs):
    for input_i, input_vector in enumerate(inputs):
        attention_scores_loop[query_i, input_i] = torch.dot(
            query_vector, input_vector
        )

print("Loop score matrix:")
print(attention_scores_loop)
print("shape:", tuple(attention_scores_loop.shape))
print("\nJourney row:")
for token, score in zip(tokens, attention_scores_loop[1]):
    print(f"journey · {token:<7} = {score.item():.4f}")

**What to notice:** row 2 contains `journey·your`, `journey·journey`, …, `journey·step`.

**Concept check:** What do the columns represent within one row?

## 28. Visualize the score matrix

Every row answers: “For this query, what are the raw compatibility scores with all tokens?”

In [ ]:
annotated_heatmap(
    attention_scores_loop.numpy(),
    tokens,
    tokens,
    "Raw dot-product attention score matrix",
    "Raw score",
    cmap="Purples",
)

**What to notice:** the matrix is symmetric because `xᵢ·xⱼ = xⱼ·xᵢ` in this simplified version.

**Concept check:** Which cell represents `journey·starts`?

## 29. Replace two loops with matrix multiplication

If `X` has shape `(6, 3)`, then:

```text
(6 × 3) @ (3 × 6) = (6 × 6)
     X       Xᵀ          S
```

Entry `(X Xᵀ)[i, j]` is `xᵢ·xⱼ`.

In [ ]:
attention_scores = inputs @ inputs.T

print("Matrix score shape:", tuple(attention_scores.shape))
print("Matrix and loops match:", torch.allclose(attention_scores, attention_scores_loop))
print("Journey row matches earlier scores:",
      torch.allclose(attention_scores[1], attention_scores_2))

**What to notice:** one optimized tensor operation reproduces all 36 dot products.

**Concept check:** Why must the inner dimensions both be 3?

## 30. Why linear algebra matters

Nested Python loops are easy to study but slow at scale. Matrix multiplication performs the same mathematical work using optimized, vectorized CPU/GPU operations.

This is a practical speed advantage, not a claim of lower big-O complexity.

**Concept check:** Does vectorization change the dot products being calculated?

## 31. Normalize the entire score matrix

Apply softmax across the last dimension so each query row becomes one normalized distribution.

In [ ]:
attention_weights = torch.softmax(attention_scores, dim=-1)
row_sums = attention_weights.sum(dim=-1)

print("Attention weights:")
print(attention_weights)
print("\nshape:", tuple(attention_weights.shape))
print("row sums:", row_sums)
print("all rows sum to 1:", torch.allclose(row_sums, torch.ones_like(row_sums)))

**What to notice:** every row is normalized independently.

**Concept check:** Would `softmax(dim=0)` normalize the intended groups here?

## 32. Understanding `dim=-1`

For a `(6, 6)` matrix:

- `dim=0`: move down rows for each column.
- `dim=1`: move across columns within each row.
- `dim=-1`: use the last dimension, which is equivalent to `dim=1` here.

Therefore `softmax(dim=-1)` normalizes across candidate tokens for each query.

In [ ]:
toy_matrix = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
    [7.0, 8.0, 9.0],
])
toy_row_softmax = torch.softmax(toy_matrix, dim=-1)

fig, ax = plt.subplots(figsize=(5.8, 4.5))
ax.imshow(np.zeros((3, 3)), cmap="Greys", vmin=0, vmax=1)
for row in range(3):
    for col in range(3):
        face = "#fde68a" if row == 1 else "#e5e7eb"
        ax.text(col, row, f"{toy_matrix[row, col].item():.0f}",
                ha="center", va="center",
                bbox=dict(boxstyle="round", fc=face, ec="none"))
ax.set_xticks(range(3), ["column 0", "column 1", "column 2"])
ax.set_yticks(range(3), ["row 0", "row 1", "row 2"])
ax.set_xlabel("dim=-1 moves across these columns")
ax.set_ylabel("Rows are separate groups")
ax.set_title("One row is one softmax group")
plt.show()

print("softmax rows:\n", toy_row_softmax)
print("row sums:", toy_row_softmax.sum(dim=-1))

**What to notice:** the highlighted middle row is normalized as one group; it is not mixed with values above or below it.

**Concept check:** For a 3D tensor, would `dim=-1` still always mean `dim=1`?

## 33. Attention-weight matrix heatmap

This heatmap shows normalized weights, not raw scores.

In [ ]:
annotated_heatmap(
    attention_weights.numpy(),
    tokens,
    tokens,
    "Row-normalized attention weight matrix",
    "Attention weight",
    cmap="Blues",
    value_format=".3f",
)

**Illustrative values for learning — not weights learned by a real LLM.**

**What to notice:** every row is a probability-like distribution over the sequence.

**Concept check:** What should the six numbers in the *step* row sum to?

## 34. Context vector for every token

```text
attention_weights @ inputs = context_vectors
     (6 × 6)       (6 × 3)      (6 × 3)
```

Each output row is one 3D context vector.

In [ ]:
context_vectors = attention_weights @ inputs

print_table(
    ["token", "context dim 1", "context dim 2", "context dim 3"],
    [
        [token, *(f"{value:.4f}" for value in vector.tolist())]
        for token, vector in zip(tokens, context_vectors)
    ],
)
print("\ncontext_vectors.shape:", tuple(context_vectors.shape))

**What to notice:** attention changes the values but preserves one 3D row per token.

**Concept check:** Which dimension still represents sequence length?

## 35. Why `A @ X` works

Take only the *journey* row:

```text
[a1, a2, a3, a4, a5, a6] @ X

= a1*x1 + a2*x2 + a3*x3 + a4*x4 + a5*x5 + a6*x6
```

Matrix multiplication performs this weighted sum for every query row simultaneously.

In [ ]:
print("Journey row weighted sum:")
for token, weight, vector in zip(tokens, attention_weights[1], inputs):
    print(
        f"{weight.item():.4f} × {token:<7} {vector.tolist()} "
        f"= {(weight * vector).tolist()}"
    )
print("\nsum =", (attention_weights[1, :, None] * inputs).sum(dim=0))

**What to notice:** the row selects one set of six coefficients; the columns of `X` determine the three output dimensions.

**Concept check:** Does `A @ X` mix embedding dimensions with query rows?

## 36. Verify the journey result

The earlier loop and the matrix calculation should produce the same `z₂`.

In [ ]:
print("manual z₂:", context_vec_2)
print("matrix z₂:", context_vectors[1])
print("match:", torch.allclose(context_vec_2, context_vectors[1]))

**What to notice:** `True` confirms that the compact matrix expression preserves the step-by-step calculation.

**Concept check:** Which version is easier to inspect, and which is easier to scale?

## 37. The whole simplified attention pipeline

```text
FOR ONE QUERY

input embeddings X
       ↓
choose query xᵢ
       ↓
dot product with every xⱼ
       ↓
scores → softmax → weights
       ↓
weighted sum of xⱼ
       ↓
context vector zᵢ

MATRIX FORM

X → S = X Xᵀ → A = row-softmax(S) → Z = A X
```

> **Simplified self-attention without trainable weights**

**Concept check:** Which matrix contains the normalized relationships?

## 38. Shape tracking

| Object | Meaning | This example | General |
|---|---|---:|---:|
| `X` | input embeddings | `6 × 3` | `n × d` |
| `Xᵀ` | transposed inputs | `3 × 6` | `d × n` |
| `S = X Xᵀ` | attention scores | `6 × 6` | `n × n` |
| `A = softmax(S)` | attention weights | `6 × 6` | `n × n` |
| `Z = A X` | context vectors | `6 × 3` | `n × d` |

`n` is sequence length; `d` is embedding dimension.

**Concept check:** If `X` is `8 × 4`, what shape is `S`?

## 39. Visual attention explorer

The function below avoids an `ipywidgets` dependency. Choose any token index to inspect its query, scores, weights, highlighted matrix row, and context.

In [ ]:
def visualize_query(token_index):
    if token_index not in range(len(tokens)):
        raise ValueError(f"token_index must be between 0 and {len(tokens)-1}")

    token = tokens[token_index]
    scores = attention_scores[token_index]
    weights = attention_weights[token_index]
    context = context_vectors[token_index]

    print(f"query token: {token}")
    print("query vector:", inputs[token_index])
    print("context vector:", context)

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
    axes[0].bar(tokens, scores.numpy(), color=COLORS["purple"])
    axes[0].set(title="Raw scores", xlabel="Compared token", ylabel="Score")
    axes[1].bar(tokens, weights.numpy(), color=COLORS["orange"])
    axes[1].set(title="Softmax weights", xlabel="Contributing token", ylabel="Weight")
    axes[2].imshow(attention_weights.numpy(), cmap="Blues", aspect="auto")
    axes[2].axhspan(token_index - 0.5, token_index + 0.5,
                    facecolor="none", edgecolor=COLORS["red"], lw=3)
    axes[2].set_xticks(range(len(tokens)), tokens, rotation=35, ha="right")
    axes[2].set_yticks(range(len(tokens)), tokens)
    axes[2].set(title="Highlighted attention row", xlabel="Candidate", ylabel="Query")
    for ax in axes[:2]:
        ax.tick_params(axis="x", rotation=35)
    fig.suptitle(f"Attention explorer: query = {token}")
    plt.tight_layout()
    plt.show()


visualize_query(1)  # journey
visualize_query(5)  # step

**What to notice:** changing the query selects a different score row, weight row, and context vector.

**Try it:** call `visualize_query(0)` or another valid index.

## 40. Important limitation of this version

Our current score is:

\[
score(i,j) = x_i \cdot x_j
\]

Consider: **“The cat sat on the mat because it is warm.”** For the query *warm*, context may require a strong relationship with *mat* even if *mat* and *warm* are not especially similar as isolated embedding concepts.

Raw embedding geometry cannot learn which relationship the task needs in this sentence.

**Concept check:** Can a low raw similarity still hide an important contextual relationship?

## 41. Why trainable weights are needed

We want the model to **learn** which relationships matter:

```text
CURRENT
xᵢ → direct dot products

NEXT
learned transformations
         ↓
  Query, Key, Value
         ↓
learned attention relationships
```

Trainable projections can express relationships beyond raw embedding similarity. We preview them here but do not implement them.

**Concept check:** What capability do learned transformations add?

## 42. Precision note

> This notebook intentionally uses **input embedding · input embedding** as the score.

Modern Transformer self-attention does not simply calculate semantic similarity between raw embeddings. Learned Query and Key representations determine scores, while learned Value representations supply the information that is combined.

Those calculations begin in the next notebook.

## 43. Scores, weights, and context

| Attention score | Attention weight | Context vector |
|---|---|---|
| Example: `1.49` | Example: `0.238` | Example: `[0.44, 0.65, 0.57]` |
| Raw compatibility | Normalized contribution | Weighted sequence mixture |

```text
score → softmax → weight → weighted sum → context
```

**Concept check:** Which item is a vector rather than one scalar?

## 44. Softmax visual experiment

Compare how different raw patterns become normalized distributions.

In [ ]:
def show_softmax(score_sets):
    fig, axes = plt.subplots(len(score_sets), 2, figsize=(9, 2.5 * len(score_sets)))
    for row, scores_list in enumerate(score_sets):
        scores_tensor = torch.tensor(scores_list, dtype=torch.float32)
        weights_tensor = torch.softmax(scores_tensor, dim=0)
        labels = [str(i + 1) for i in range(len(scores_list))]

        axes[row, 0].bar(labels, scores_tensor.numpy(), color=COLORS["purple"])
        axes[row, 0].set(ylabel="Value", title=f"Raw: {scores_list}")
        axes[row, 1].bar(labels, weights_tensor.numpy(), color=COLORS["orange"])
        axes[row, 1].set(
            ylim=(0, 1), ylabel="Weight",
            title=f"Softmax: {[round(v, 3) for v in weights_tensor.tolist()]}",
        )
    axes[-1, 0].set_xlabel("Position")
    axes[-1, 1].set_xlabel("Position")
    fig.suptitle("Raw scores and their softmax weights", y=1.01)
    plt.tight_layout()
    plt.show()


score_sets = [
    [1, 1, 1],
    [1, 2, 3],
    [1, 2, 8],
    [-2, 0, 2],
    [1000, 999, 998],
]
show_softmax(score_sets)

**What to notice:** equal scores give equal weights; a widening score gap makes the distribution more concentrated. Large shared offsets do not break stable softmax.

**Concept check:** What happens when one score becomes much larger than the others?

## 45. Optional: temperature intuition

> **Extra intuition — not required for this lecture.**

`softmax(scores / T)` changes concentration: lower `T` is sharper; higher `T` is flatter.

In [ ]:
temperature_scores = torch.tensor([1.0, 2.0, 3.0])
temperatures = [0.5, 1.0, 2.0]

for temperature in temperatures:
    values = torch.softmax(temperature_scores / temperature, dim=0)
    plt.plot([1, 2, 3], values.numpy(), marker="o", lw=2,
             label=f"T = {temperature}")
plt.xticks([1, 2, 3])
plt.xlabel("Score position")
plt.ylabel("Softmax weight")
plt.title("Optional temperature effect")
plt.legend()
plt.show()

**What to notice:** temperature changes sharpness, not score ordering.

## 46. Common confusions

**Is the query always *journey*?** No. It is only the first demonstration; every token becomes a query.

**Why use a dot product?** It is a convenient simplified compatibility measure.

**Does a high dot product guarantee linguistic importance?** No; that is the main limitation here.

**Are score and weight the same?** No. A score is raw; a weight is normalized.

**Why softmax?** It produces positive, normalized weights while preserving score order.

**Why subtract the maximum?** For numerical stability; it does not change softmax.

**Why does each attention row sum to 1?** Softmax is applied across candidates for one query.

**What does `dim=-1` mean?** Normalize the final dimension—columns within each row here.

**Why are `S` and `A` both `6×6`?** Six queries are compared with six candidate positions.

**Why is `Z` `6×3`?** There are six tokens, each with one 3D context vector.

**Are these learned attention weights?** No trainable attention parameters exist here.

## 47. Mini numerical exercises

1. For `q=[1,2]`, calculate dot products with `x1=[1,0]`, `x2=[1,2]`, and `x3=[0,1]`.
2. Apply softmax to `[1,5,2]`.
3. For weights `[0.2,0.5,0.3]` and vectors `[1,0]`, `[0,2]`, `[2,1]`, calculate the context.
4. If `X` is `8×4`, what shape is `X @ X.T`?
5. If an `8×8` score matrix uses `softmax(dim=-1)`, what should each row sum to?
6. If `A` is `8×8` and `X` is `8×4`, what is the shape of `A @ X`?
7. Why should `softmax([1000,999])` use a stable implementation?
8. Why may raw embedding similarity miss a contextually important relationship?

Solutions appear after the implementation challenge.

## 48. Visual quiz

Match each diagram with:

**A.** raw embeddings · **B.** one score · **C.** softmax · **D.** weight matrix · **E.** weighted sum · **F.** `X @ X.T` · **G.** `A @ X`

In [ ]:
visual_cards = [
    ("Diagram 1", "X\\n[n × d]"),
    ("Diagram 2", "q · xⱼ\\n→ scalar"),
    ("Diagram 3", "scores\\n↓ softmax\\nweights"),
    ("Diagram 4", "A\\n[n × n]\\nrows sum to 1"),
    ("Diagram 5", "Σ aⱼxⱼ\\n→ zᵢ"),
    ("Diagram 6", "X @ Xᵀ\\n→ S"),
    ("Diagram 7", "A @ X\\n→ Z"),
]
fig, axes = plt.subplots(1, 7, figsize=(17, 3))
for ax, (title, body) in zip(axes, visual_cards):
    ax.text(0.5, 0.5, body, ha="center", va="center", fontsize=11,
            bbox=dict(boxstyle="round", fc="#e0e7ff", ec=COLORS["purple"]))
    ax.set_title(title)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.axis("off")
fig.suptitle("Identify each stage of simplified self-attention")
plt.tight_layout()
plt.show()

## 49. Theory quiz

1. What does a token embedding represent?
2. What does embedding dimension mean?
3. What is a context vector?
4. What is the query in this simplified example?
5. What is an attention score?
6. How is the score calculated here?
7. What does vector alignment suggest about a dot product?
8. Why is a dot product not cosine similarity?
9. What is an attention weight?
10. Why normalize scores?
11. How does simple sum normalization work?
12. Why can direct sum normalization be awkward?
13. What are the three steps of softmax?
14. Why are softmax outputs positive?
15. Why do softmax outputs sum to 1?
16. Does softmax always make one result almost 1?
17. Why can naive exponentiation overflow?
18. What is the subtract-max trick?
19. Why does subtracting max preserve softmax?
20. What is the difference between a score and a weight?
21. How is one context vector formed?
22. Why is this mechanism called self-attention?
23. Why must every token act as a query?
24. What does one score-matrix row mean?
25. Why does `X @ X.T` contain all pairwise dot products?
26. If `X` is `n×d`, what shape is the score matrix?
27. What does row-wise softmax normalize?
28. What does `dim=-1` mean here?
29. What does one attention-matrix row contain?
30. Why does `A @ X` create weighted sums?
31. What is the shape of `Z` for `X` shaped `n×d`?
32. Why are vectorized operations faster in practice?
33. What is the main limitation of direct embedding dot products?
34. How can semantic similarity differ from contextual importance?
35. Why are trainable transformations needed?
36. What roles will Query, Key, and Value introduce next?

Do not open the answer section until you have attempted the questions.

## 50. Code reconstruction challenge

Reproduce simplified self-attention in three lines. Uncomment and replace the blanks.

In [ ]:
# Step 1: attention scores
# scores_challenge = ____________________

# Step 2: attention weights
# weights_challenge = ___________________

# Step 3: context vectors
# context_challenge = ___________________

## 51. Complete end-to-end implementation

Here is the entire simplified mechanism in its minimal form.

In [ ]:
X = inputs
S = X @ X.T
A = torch.softmax(S, dim=-1)
Z = A @ X

print("X shape:", tuple(X.shape))
print("S shape:", tuple(S.shape))
print("A shape:", tuple(A.shape))
print("Z shape:", tuple(Z.shape))
print("\nS =\n", S)
print("\nA =\n", A)
print("\nZ =\n", Z)
print("\nA row sums =", A.sum(dim=-1))

assert torch.allclose(S, attention_scores)
assert torch.allclose(A, attention_weights)
assert torch.allclose(Z, context_vectors)
assert torch.allclose(A.sum(dim=-1), torch.ones(len(tokens)))

- `S = X @ X.T` calculates all raw compatibility scores.
- `A = torch.softmax(S, dim=-1)` normalizes each query row.
- `Z = A @ X` calculates all weighted context vectors.

**What to notice:** the three-line result matches every earlier loop and assertion.

## Answers — check only after attempting

<details>
<summary><strong>Numerical exercise solutions</strong></summary>

1. Scores: `[1, 5, 2]`.
2. Approximately `[0.0171, 0.9362, 0.0466]`.
3. `[0.2,0] + [0,1.0] + [0.6,0.3] = [0.8,1.3]`.
4. `8×8`.
5. Approximately `1`.
6. `8×4`.
7. Direct exponentials can overflow; subtract the maximum or use `torch.softmax`.
8. The relationship required by the current task may differ from isolated-vector similarity.
</details>

<details>
<summary><strong>Visual quiz answers</strong></summary>

1. A — raw embeddings  
2. B — one score calculation  
3. C — softmax normalization  
4. D — attention-weight matrix  
5. E — context-vector weighted sum  
6. F — `X @ X.T`  
7. G — `A @ X`
</details>

<details>
<summary><strong>Theory quiz answers</strong></summary>

1. A token's numerical starting representation.
2. The number of values used per token vector.
3. A weighted mixture containing sequence information for one query.
4. The representation whose context is being calculated.
5. Raw query–candidate compatibility.
6. A direct dot product of input vectors.
7. Closer direction generally increases it.
8. Magnitudes also affect a dot product.
9. A normalized relative contribution.
10. To create controlled relative contributions.
11. Divide each positive score by their total.
12. Negative or zero totals can produce unsuitable results.
13. Exponentiate, sum exponentials, divide by the sum.
14. Exponentials are positive.
15. Each exponential is divided by their total.
16. No; concentration depends on score gaps.
17. Very large exponentials exceed numeric range.
18. Subtract the largest score before exponentiating.
19. The common exponential factor cancels.
20. Score is raw; weight is normalized.
21. Sum every input vector multiplied by its weight.
22. Queries and candidates come from the same sequence.
23. Every token needs its own contextual result.
24. One query's scores against all positions.
25. Matrix entry `[i,j]` is row `i` dotted with row `j`.
26. `n×n`.
27. Candidates within each query row.
28. The last dimension—columns here.
29. One normalized distribution over sequence positions.
30. Each row of `A` supplies coefficients for rows of `X`.
31. `n×d`.
32. Optimized tensor kernels exploit parallel hardware.
33. Scores depend only on raw embedding geometry.
34. A dissimilar token may still be important in the sentence.
35. So the model can learn task-relevant relationships.
36. Learned scoring representations and learned information to combine; details come next.
</details>

<details>
<summary><strong>Code challenge solution</strong></summary>

```python
scores_challenge = X @ X.T
weights_challenge = torch.softmax(scores_challenge, dim=-1)
context_challenge = weights_challenge @ X
```
</details>

## 52. Final concept map

```text
Sentence → tokens → token embeddings X
                      ↓
            each token becomes query
                      ↓
                 dot products
                      ↓
             attention scores S
                      ↓ softmax
             attention weights A
                      ↓ weighted combination
              context vectors Z

S = X Xᵀ       A = softmax(S)       Z = A X

LIMITATION: scores depend directly on raw embedding geometry
                      ↓
NEXT: trainable transformations → Query, Key, Value
```

**Concept check:** Which step turns six input rows into 36 pairwise scores?

## 53. “Can I explain this?” checklist

- [ ] why embeddings alone are not enough
- [ ] embedding vector, query, score, weight, and context vector
- [ ] a dot product by hand
- [ ] dot product vs cosine similarity
- [ ] why scores need normalization
- [ ] simple normalization and its limitation
- [ ] softmax by hand
- [ ] positivity, sum-to-one, and relative emphasis
- [ ] subtract-max stability and why it preserves the result
- [ ] a context vector as a weighted sum
- [ ] why every token acts as a query
- [ ] why `S` and `A` are `n×n`
- [ ] how `X @ X.T` creates scores
- [ ] row-wise softmax and `dim=-1`
- [ ] how `A @ X` creates `n×d` context vectors
- [ ] simplified self-attention in three lines
- [ ] why this version has no trainable attention weights
- [ ] the limitation of raw embedding geometry
- [ ] why trainable Query/Key/Value projections come next

## 54. Final summary

**Simplified self-attention**

1. Compare one query with every token: **dot products → attention scores**
2. Normalize the scores: **softmax → attention weights**
3. Combine information: **weighted sum → context vector**

For the whole sequence:

\[
S = X X^T,\qquad A = softmax(S),\qquad Z = A X
\]

**What we achieved:** an embedding becomes a context-aware mixture of sequence information.

**What is missing:** the mechanism cannot yet learn which kinds of relationships should matter.

---

## Next notebook

# Self-Attention with Trainable Weights

Query · Key · Value

This notebook ends exactly at that motivation.